# 📘 Session 4B: Regular Expressions — Comprehensive Guide
### Duration: ~2–2.5 Hours

---

**Topics Covered:**
1. What Are Regular Expressions? — Mental model for beginners
2. Python's `re` Module — Raw strings and first matches
3. Literal Characters vs Metacharacters
4. Character Classes — `[ ]`, ranges, and negation
5. Shorthand Classes — `\d`, `\w`, `\s` and friends
6. Quantifiers — `*`, `+`, `?`, `{n,m}` and greedy vs lazy
7. Anchors & Word Boundaries — `^`, `$`, `\b`
8. Grouping & Capturing — `()` and extracting parts
9. The `re` Toolkit — `search`, `match`, `findall`, `sub`, `split`, `compile`
10. Match Objects — `group()`, `groups()`, `span()`
11. Flags — `IGNORECASE`, `MULTILINE`, `DOTALL`, `VERBOSE`
12. Alternation, Backreferences & Lookahead
13. Real-World Patterns — emails, phones, URLs, logs, dates
14. Regex in Data Science & NLP
15. Regex vs String Methods — When to use what
16. Debugging & Cheat Sheet
17. Practice Exercises

---

**Prerequisites:** Sessions 1–4 (especially strings and the regex basics in Session 4)

**Why This Session Exists:**
Session 4 introduced regex in ~15 minutes. Students asked for a **dedicated deep dive** — this session builds regex from zero so anyone can understand it, even with no prior exposure.

> **Data Science relevance:** Cleaning messy text, parsing log files, validating user input, extracting features from unstructured data, and preprocessing text before NLP — regex is a daily tool for data professionals.


---
## 0. What Is a Regular Expression?

A **regular expression** (regex) is a **mini-language for describing text patterns**.

### Everyday Analogy

Imagine you are told: *"Find every phone number in this 500-page document."*

| Approach | How it works | Problem |
|----------|--------------|---------|
| **Manual search** | Look for `555` | Misses `(555) 123-4567`, `555.123.4567`, etc. |
| **String methods** | `if '-' in line` | Too rigid — misses valid variations |
| **Regex** | Pattern like `\d{3}[-.]?\d{3}[-.]?\d{4}` | Matches **all common formats** at once |

Regex answers: *"Does this text **look like** the pattern I described?"*

### What Regex Is Good At

- ✅ Finding patterns: emails, dates, prices, IP addresses
- ✅ Extracting parts: area code, domain name, year from a date
- ✅ Replacing patterns: mask emails, normalize phone formats
- ✅ Validating input: "Is this a valid Indian pin code?"
- ✅ Splitting text: break on multiple delimiters (`;`, `,`, `|`)

### What Regex Is NOT Good At

- ❌ Parsing nested structures (HTML, JSON) — use proper parsers
- ❌ Remembering context across very long documents — use NLP/ML
- ❌ "Understanding" meaning — regex matches **shape**, not **semantics**

### The Regex Engine (Simple Mental Model)

```
Your text:  "Order #1234 on 2024-01-15"
Your pattern: r'\d{4}-\d{2}-\d{2}'

Engine walks through text left-to-right:
  O → no match... r → no match... 2 → start trying pattern...
  2024-01-15 → MATCH! ✓
```

The engine tries your pattern at each position until it finds a match (for `search`) or confirms the whole string matches (for `fullmatch`).

> 💡 **Tip:** Think of regex as a **smart find-and-replace** in Word/Google Docs — but programmable and far more powerful.


In [1]:
# --- Your First Regex (run this cell!) ---
import re

text = "My email is alice@example.com and my backup is bob@company.co.uk"

# Does the text contain something that LOOKS like an email?
pattern = r'[\w.]+@[\w.]+\.[a-zA-Z]{2,}'

match = re.search(pattern, text)
if match:
    print(f"Found: '{match.group()}'")
    print(f"At position: {match.span()}")  # (start, end)
else:
    print("No match")

# findall finds ALL matches
all_emails = re.findall(pattern, text)
print(f"All emails: {all_emails}")


Found: 'alice@example.com'
At position: (12, 29)
All emails: ['alice@example.com', 'bob@company.co.uk']


---
## 1. Python Setup — The `re` Module & Raw Strings

```python
import re
```

Python's built-in `re` module implements regex. No pip install needed.

### Always Use Raw Strings: `r"..."`

Backslashes are special in **both** Python strings **and** regex. Raw strings pass backslashes to regex unchanged.

| Code | What Python sees | What regex engine sees |
|------|------------------|------------------------|
| `"\d+"` | `\d+` | ✅ digit pattern |
| `"\d+"` (single backslash) | `\d+` after escape — **broken!** | ❌ |
| `r"\d+"` | `\d+` literally | ✅ digit pattern |

```python
# ❌ Confusing — double escaping
re.search("\\d+", "abc123")

# ✅ Clear — always use raw strings for regex
re.search(r"\d+", "abc123")
```

> ⚠️ **Special Case:** Raw strings still need `r"\"` for a literal backslash in the pattern. The `r` prefix only affects Python's interpretation, not regex rules.


In [2]:
# --- Raw String Demo ---
import re

text = "Room 42, Floor 7"

# Without raw string — you need double backslashes (error-prone)
print(re.findall("\\d+", text))   # works but ugly

# With raw string — clean and readable
print(re.findall(r"\d+", text))    # ['42', '7']

# Windows path example (why raw strings matter beyond regex)
path_normal = "C:\\Users\\data"   # need 4 backslashes!
path_raw    = r"C:\Users\data"       # clean


['42', '7']
['42', '7']


---
## 2. Literal Characters vs Metacharacters

Most characters in a regex match **themselves** (literals).

| Pattern | Text | Match? |
|---------|------|--------|
| `cat` | `"the cat sat"` | ✅ matches `cat` |
| `cat` | `"catalog"` | ✅ matches `cat` at start |
| `cat` | `"CAT"` | ❌ case-sensitive by default |

**Metacharacters** have special meaning — they are the "power tools" of regex:

| Metacharacter | Meaning |
|---------------|---------|
| `.` | Any single character (except newline by default) |
| `^` | Start of string (or line with MULTILINE) |
| `$` | End of string (or line with MULTILINE) |
| `*` | Zero or more of the preceding |
| `+` | One or more of the preceding |
| `?` | Zero or one of the preceding |
| `{n,m}` | Between n and m of the preceding |
| `[ ]` | Character class — any one of these |
| `( )` | Grouping / capturing |
| `\|` | OR — match left or right |
| `\` | Escape a metacharacter to make it literal |

### Escaping — Making Metacharacters Literal

To match a literal `.`, `*`, `?`, etc., put `\` before it:

| Pattern | Matches |
|---------|---------|
| `r"3\.14"` | `3.14` (literal dot) |
| `r"\$99"` | `$99` (literal dollar sign) |
| `r"\*"` | `*` (literal asterisk) |

> ⚠️ **Special Case — The dot `.`**: In regex, `.` means "any character". To match a literal period in `file.txt`, use `r"file\.txt"`.


In [3]:
# --- Literals vs Metacharacters ---
import re

# Literal match
print(re.search(r"cat", "the catalog"))       # Match object — 'cat' found
print(re.search(r"cat", "The CAT"))            # None — case sensitive

# Dot matches ANY character (except newline)
print(re.findall(r"c.t", "cat cot cut cit"))   # ['cat', 'cot', 'cut', 'cit']

# Escape dot for literal match
print(re.search(r"3\.14", "pi is 3.14"))      # Match
print(re.search(r"3\.14", "pi is 3x14"))       # None

# Escape special characters
price_text = "Items cost $19.99 and $5.00"
prices = re.findall(r"\$[\d.]+", price_text)
print(f"Prices: {prices}")


<re.Match object; span=(4, 7), match='cat'>
None
['cat', 'cot', 'cut', 'cit']
<re.Match object; span=(6, 10), match='3.14'>
None
Prices: ['$19.99', '$5.00']


---
## 3. Character Classes — `[ ]`

A **character class** matches **exactly one** character from a set you define.

| Pattern | Matches | Does NOT match |
|---------|---------|----------------|
| `[aeiou]` | any vowel | consonants, digits |
| `[0-9]` | any digit | letters |
| `[a-z]` | lowercase a–z | uppercase, digits |
| `[A-Za-z]` | any letter | digits, symbols |
| `[a-zA-Z0-9_]` | same as `\w` | spaces, punctuation |

### Ranges Inside `[ ]`

```python
r"[0-9]"      # digits 0 through 9
r"[a-f]"      # a, b, c, d, e, f  (hex digits)
r"[2-9]"      # digits 2 through 9 (phone keypad)
```

### Negation — `[^ ]`

`[^...]` means **"any character EXCEPT these"**:

| Pattern | Matches |
|---------|---------|
| `[^0-9]` | anything that is NOT a digit |
| `[^aeiou]` | any non-vowel |
| `[^\n]` | anything except newline |

> ⚠️ **Special Case:** Inside `[ ]`, most metacharacters lose special meaning. `[.]` matches a literal dot. `[*+?]` are literals too. Only `^` (at start), `-` (between two chars), and `\` stay special.


In [4]:
# --- Character Classes ---
import re

text = "Hello World 123!"

print("Vowels:", re.findall(r"[aeiouAEIOU]", text))
print("Digits:", re.findall(r"[0-9]", text))
print("Letters:", re.findall(r"[A-Za-z]", text))
print("Non-digits:", re.findall(r"[^0-9]", text))  # includes spaces & !

# Hex color codes
colors = "#FF5733 #abc #GGGGGG #00FF00"
valid_hex = re.findall(r"#[0-9A-Fa-f]{6}", colors)
print(f"Valid hex colors: {valid_hex}")

# Indian vehicle number plate (simplified): KA01AB1234
plates = "KA01AB1234 MH12DE5678 invalid123"
plate_pattern = r"[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}"
print("Plates:", re.findall(plate_pattern, plates))


Vowels: ['e', 'o', 'o']
Digits: ['1', '2', '3']
Letters: ['H', 'e', 'l', 'l', 'o', 'W', 'o', 'r', 'l', 'd']
Non-digits: ['H', 'e', 'l', 'l', 'o', ' ', 'W', 'o', 'r', 'l', 'd', ' ', '!']
Valid hex colors: ['#FF5733', '#00FF00']
Plates: ['KA01AB1234', 'MH12DE5678']


---
## 4. Shorthand Character Classes

These save typing for common character sets:

| Shorthand | Same as | Matches |
|-----------|---------|---------|
| `\d` | `[0-9]` | Any digit |
| `\D` | `[^0-9]` | Any non-digit |
| `\w` | `[a-zA-Z0-9_]` | Word character |
| `\W` | `[^a-zA-Z0-9_]` | Non-word character |
| `\s` | `[ \t\n\r\f\v]` | Whitespace |
| `\S` | `[^\s]` | Non-whitespace |

### Quick Reference Examples

```python
r"\d+"      # one or more digits: "123", "42"
r"\w+"      # one or more word chars: "hello", "test_1"
r"\s+"      # one or more whitespace chars
r"\D+"      # one or more non-digits
```

> **Unicode note:** In Python 3, `\w` also matches Unicode letters (é, ñ, 中) when using the default `re` module. For ASCII-only matching, use `re.ASCII` flag.


In [5]:
# --- Shorthand Classes ---
import re

sample = "User: alice42 scored 95.5 points!"

print("Words:", re.findall(r"\w+", sample))
print("Numbers:", re.findall(r"\d+\.?\d*", sample))
print("Non-word chunks:", re.findall(r"\W+", sample))

# Split on any whitespace
text = "apple\tbanana  cherry\n  date"
print(re.split(r"\s+", text.strip()))


Words: ['User', 'alice42', 'scored', '95', '5', 'points']
Numbers: ['42', '95.5']
Non-word chunks: [': ', ' ', ' ', '.', ' ', '!']
['apple', 'banana', 'cherry', 'date']


---
## 5. Quantifiers — How Many Times?

Quantifiers control **how many times** the preceding element repeats.

| Quantifier | Meaning | Example | Matches |
|------------|---------|---------|---------|
| `*` | 0 or more | `r"ab*c"` | `ac`, `abc`, `abbc` |
| `+` | 1 or more | `r"ab+c"` | `abc`, `abbc` — NOT `ac` |
| `?` | 0 or 1 | `r"colou?r"` | `color`, `colour` |
| `{n}` | exactly n | `r"\d{4}"` | `2024` |
| `{n,}` | n or more | `r"\d{2,}"` | `42`, `1234` |
| `{n,m}` | between n and m | `r"\d{2,4}"` | `42`, `123`, `1234` |

### Greedy vs Lazy (Non-Greedy)

By default, quantifiers are **greedy** — they match as much as possible.

| Type | Syntax | Behavior |
|------|--------|----------|
| Greedy (default) | `*`, `+`, `?`, `{n,m}` | Match as **much** as possible |
| Lazy (non-greedy) | `*?`, `+?`, `??`, `{n,m}?` | Match as **little** as possible |

```python
text = "<b>bold</b> and <i>italic</i>"

r"<.*>"    # Greedy:  "<b>bold</b> and <i>italic</i>"  (whole string!)
r"<.*?>"   # Lazy:     "<b>" then "</b>" then "<i>" then "</i>"
```

> ⚠️ **Special Case — Greedy traps:** `r".*"` on a long string can swallow everything. Use lazy quantifiers `.*?` or more specific patterns when extracting HTML/XML tags.


In [6]:
# --- Quantifiers ---
import re

# Basic quantifiers
print(re.findall(r"ab*c", "ac abc abbc abbbc"))   # all match
print(re.findall(r"ab+c", "ac abc abbc"))         # 'ac' excluded
print(re.findall(r"colou?r", "color colour"))     # both

# Exact counts
print(re.findall(r"\d{4}", "Year 2024, code 12345"))  # ['2024', '1234'] — partial from 12345

# Greedy vs Lazy
html = "<b>bold</b> and <i>italic</i>"
print("Greedy:", re.findall(r"<.*>", html))
print("Lazy:  ", re.findall(r"<.*?>", html))

# Phone number variations
phones = "Call 555-1234 or 555-123-4567 or 5551234567"
pattern = r"\d{3}[-.]?\d{3}[-.]?\d{4}"
print("Phones:", re.findall(pattern, phones))


['ac', 'abc', 'abbc', 'abbbc']
['abc', 'abbc']
['color', 'colour']
['2024', '1234']
Greedy: ['<b>bold</b> and <i>italic</i>']
Lazy:   ['<b>', '</b>', '<i>', '</i>']
Phones: ['555-123-4567', '5551234567']


---
## 6. Anchors & Word Boundaries

Anchors don't match characters — they match **positions** in the text.

| Anchor | Position |
|--------|----------|
| `^` | Start of string (or start of line with `re.MULTILINE`) |
| `$` | End of string (or end of line with `re.MULTILINE`) |
| `\b` | Word boundary (between `\w` and `\W`) |
| `\B` | Non-word boundary |

### Why Anchors Matter

```python
r"cat"     # matches "cat" anywhere: "the cat", "catalog"
r"^cat"    # only at start: "catalog" ✅, "the cat" ❌
r"cat$"    # only at end: "the cat" ✅, "catalog" ❌
r"^cat$"   # entire string must be exactly "cat"
```

### Word Boundaries — Avoiding Partial Matches

```python
r"\bcat\b"   # matches "the cat sat" but NOT "catalog" or "scatter"
```

> **Validation pattern:** Use `^` and `$` together to validate entire strings:
> `r"^[\w.]+@[\w]+\.[a-z]{2,}$"` — the whole string must be an email-like pattern.


In [7]:
# --- Anchors & Boundaries ---
import re

# ^ and $ for full-string validation
def is_valid_username(name):
    return bool(re.fullmatch(r"[a-zA-Z][a-zA-Z0-9_]{2,15}", name))

tests = ["alice", "42bad", "ab", "valid_user_1", "way_too_long_username_here"]
for t in tests:
    status = "✅" if is_valid_username(t) else "❌"
    print(f"  {t:30s} {status}")

# Word boundaries
text = "the cat scattered catalog data"
print("\ncat anywhere:", re.findall(r"cat", text))
print("cat as word:  ", re.findall(r"\bcat\b", text))

# Multiline — ^ matches start of EACH line
log = "INFO ok\nERROR fail\nINFO ok2"
print("Lines starting with ERROR:", re.findall(r"^ERROR.*", log, re.MULTILINE))


  alice                          ✅
  42bad                          ❌
  ab                             ❌
  valid_user_1                   ✅
  way_too_long_username_here     ❌

cat anywhere: ['cat', 'cat', 'cat']
cat as word:   ['cat']
Lines starting with ERROR: ['ERROR fail']


---
## 7. Grouping & Capturing — `()`

Parentheses `()` do two things:
1. **Group** — treat multiple parts as one unit for quantifiers
2. **Capture** — remember matched text for later extraction

### Grouping for Quantifiers

```python
r"(ab)+"     # matches "ab", "abab", "ababab" — NOT "aba"
r"ab+"       # matches "ab", "abb", "abbb" — different meaning!
```

### Capturing — Extracting Parts

```python
r"(\d{4})-(\d{2})-(\d{2})"   # captures year, month, day separately
```

| Method | Returns |
|--------|---------|
| `match.group()` | entire match |
| `match.group(1)` | first capture group |
| `match.group(2)` | second capture group |
| `match.groups()` | tuple of all groups |
| `match.groupdict()` | dict (if named groups used) |

### Named Groups — `(?P<name>...)`

```python
pattern = r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})"
match.group("year")   # cleaner than group(1)
```

### Non-Capturing Groups — `(?:...)`

Use when you need grouping but don't need to extract:

```python
r"(?:https?://)"   # groups for quantifier, but doesn't create capture group
```


In [8]:
# --- Grouping & Capturing ---
import re

# Grouping changes meaning
print(re.findall(r"(ha)+", "hahaha"))    # ['ha'] — last repetition
print(re.findall(r"(ha)+", "hahaha ha"))  # ['ha', 'ha']

# Date extraction with groups
text = "Events on 2024-01-15 and 2025-06-30"
pattern = r"(\d{4})-(\d{2})-(\d{2})"
for match in re.finditer(pattern, text):
    year, month, day = match.groups()
    print(f"  {year}-{month}-{day} → year={year}, month={month}, day={day}")

# Named groups
log = "2024-01-15 14:30:22 ERROR Database timeout"
pat = r"(?P<date>\S+) (?P<time>\S+) (?P<level>\w+) (?P<message>.+)"
m = re.match(pat, log)
if m:
    print(f"\nLog: {m.groupdict()}")


['ha']
['ha', 'ha']
  2024-01-15 → year=2024, month=01, day=15
  2025-06-30 → year=2025, month=06, day=30

Log: {'date': '2024-01-15', 'time': '14:30:22', 'level': 'ERROR', 'message': 'Database timeout'}


---
## 8. The `re` Toolkit — All Key Functions

| Function | What it does | Returns |
|----------|--------------|---------|
| `re.search(pat, text)` | First match **anywhere** in text | Match or `None` |
| `re.match(pat, text)` | Match only at **start** of text | Match or `None` |
| `re.fullmatch(pat, text)` | **Entire** string must match | Match or `None` |
| `re.findall(pat, text)` | **All** matches as list of strings | `list` |
| `re.finditer(pat, text)` | **All** matches as iterator of Match objects | iterator |
| `re.sub(pat, repl, text)` | Replace matches with `repl` | new string |
| `re.split(pat, text)` | Split text at each match | `list` |
| `re.compile(pat)` | Pre-compile pattern for reuse | Pattern object |

### `search` vs `match` vs `fullmatch`

```python
text = "The price is $42"

re.search(r"\d+", text)      # ✅ finds "42"
re.match(r"\d+", text)       # ❌ doesn't start with digit
re.fullmatch(r"\d+", text)   # ❌ whole string isn't just digits
re.fullmatch(r".*\d+", text) # ✅ whole string matches pattern
```

### `findall` vs `finditer`

- `findall` → simple list, easy to use
- `finditer` → Match objects with `.group()`, `.span()` — use when you need positions

> ⚠️ **Special Case — Groups in findall:** If your pattern has **capture groups**, `findall` returns groups, not the full match!
> `re.findall(r"(\d+)-(\d+)", "1-2 3-4")` → `[('1','2'), ('3','4')]`
> To get full matches, use non-capturing groups or `finditer`.


In [9]:
# --- re Functions Comparison ---
import re

text = "Call 555-1234 or 555-5678 today"

# search — first match anywhere
m = re.search(r"\d{3}-\d{4}", text)
print(f"search: {m.group()} at {m.span()}")

# findall — all simple matches
print(f"findall: {re.findall(r'\d{3}-\d{4}', text)}")

# finditer — all matches with details
for m in re.finditer(r"\d{3}-\d{4}", text):
    print(f"  found '{m.group()}' at chars {m.start()}-{m.end()}")

# sub — replace
print(f"sub: {re.sub(r'\d{3}-\d{4}', '[PHONE]', text)}")

# split — on any digit sequence
print(f"split: {re.split(r'\d+', 'a1b22c333d')}")

# fullmatch — validation
print(f"valid code: {bool(re.fullmatch(r'[A-Z]{3}-\d{4}', 'ABC-1234'))}")
print(f"valid code: {bool(re.fullmatch(r'[A-Z]{3}-\d{4}', 'abc-1234'))}")


search: 555-1234 at (5, 13)
findall: ['555-1234', '555-5678']
  found '555-1234' at chars 5-13
  found '555-5678' at chars 17-25
sub: Call [PHONE] or [PHONE] today
split: ['a', 'b', 'c', 'd']
valid code: True
valid code: False


---
## 9. Match Objects — Working with Results

When `search`, `match`, or `fullmatch` succeed, they return a **Match object**:

```python
match = re.search(r"(\d+)", "Room 42")
match.group()    # '42' — the matched text
match.group(1)   # '42' — first capture group
match.start()    # 5 — start index in original string
match.end()      # 7 — end index (exclusive)
match.span()     # (5, 7) — tuple of start, end
```

### Always Check Before Using

```python
match = re.search(pattern, text)
if match:
    print(match.group())
else:
    print("Not found")
```

### `finditer` for Multiple Rich Matches

```python
for m in re.finditer(r"\$([\d.]+)", text):
    print(f"Price ${m.group(1)} at position {m.start()}")
```


In [10]:
# --- Match Objects ---
import re

text = "Prices: $19.99, $5.00, and $150.00"

for m in re.finditer(r"\$(\d+\.\d{2})", text):
    print(f"  ${m.group(1):>6s}  span={m.span()}  before='{text[m.start()-2:m.start()]}'")

# No match returns None — always check!
result = re.search(r"\d{10}", "no long numbers here")
print(f"\nResult when no match: {result}")
print(f"Safe check: {result.group() if result else 'Not found'}")


  $ 19.99  span=(8, 14)  before=': '
  $  5.00  span=(16, 21)  before=', '
  $150.00  span=(27, 34)  before='d '

Result when no match: None
Safe check: Not found


---
## 10. Flags — Modifying Regex Behavior

Pass flags as the last argument, or combine with `|`:

```python
re.search(pattern, text, re.IGNORECASE)
re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
```

| Flag | Short | Effect |
|------|-------|--------|
| `re.IGNORECASE` | `re.I` | `A` matches `a` |
| `re.MULTILINE` | `re.M` | `^` and `$` match line boundaries |
| `re.DOTALL` | `re.S` | `.` matches newlines too |
| `re.VERBOSE` | `re.X` | Allow comments & whitespace in pattern |
| `re.ASCII` | `re.A` | `\w`, `\d`, `\s` are ASCII-only |

### VERBOSE — Readable Complex Patterns

```python
email_pat = re.compile(r'''
    [\w.+]+      # username part
    @             # at symbol
    [\w.]+       # domain
    \.            # dot
    [a-z]{2,}     # TLD
''', re.VERBOSE | re.IGNORECASE)
```


In [11]:
# --- Flags ---
import re

text = "Python is GREAT\npython is fun\nPYTHON rocks"

# IGNORECASE
print("Case-insensitive:", re.findall(r"python", text, re.IGNORECASE))

# MULTILINE — ^ matches each line start
print("Lines with 'python':", re.findall(r"^python.*", text, re.IGNORECASE | re.MULTILINE))

# DOTALL — dot matches newlines
block = "start <tag>content\nmore content</tag> end"
print("With DOTALL:", re.search(r"<tag>.*</tag>", block, re.DOTALL).group())

# VERBOSE pattern
phone_pat = re.compile(r'''
    \b
    \(?(\d{3})\)?   # area code
    [-.\s]?           # separator
    (\d{3})           # prefix
    [-.\s]?           # separator
    (\d{4})           # line number
    \b
''', re.VERBOSE)

for m in phone_pat.finditer("Call (555) 123-4567 or 555.987.6543"):
    print(f"  Phone: ({m.group(1)}) {m.group(2)}-{m.group(3)}")


Case-insensitive: ['Python', 'python', 'PYTHON']
Lines with 'python': ['Python is GREAT', 'python is fun', 'PYTHON rocks']
With DOTALL: <tag>content
more content</tag>
  Phone: (555) 123-4567
  Phone: (555) 987-6543


---
## 11. Alternation, Backreferences & Lookahead

### Alternation — `|`

Match **one of** several patterns:

```python
r"cat|dog"           # matches "cat" OR "dog"
r"(cat|dog)s?"       # "cat", "cats", "dog", "dogs"
r"https?://"         # "http://" or "https://"
```

### Backreferences — `\1`, `\2`

Refer to a **previous capture group** in the same pattern:

```python
r"(\w+) \1"        # matches "hello hello" (doubled word)
```

In `re.sub`, use `\1` in replacement to reuse captured text:

```python
re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", "2024-01-15")
# → "15/01/2024"
```

### Lookahead & Lookbehind (Zero-Width Assertions)

These check **without consuming** characters:

| Syntax | Name | Meaning |
|--------|------|---------|
| `(?=...)` | Positive lookahead | followed by ... |
| `(?!...)` | Negative lookahead | NOT followed by ... |
| `(?<=...)` | Positive lookbehind | preceded by ... |
| `(?<!...)` | Negative lookbehind | NOT preceded by ... |

```python
r"\d+(?= dollars)"   # digits only if followed by " dollars"
r"\$\d+"            # price with $ sign (lookbehind alternative)
r"(?<!\d)\b\d+\b" # standalone numbers, not part of larger number
```

> **Beginner tip:** Lookahead/lookbehind are advanced. Learn the basics first — come back to these when you need "match X only when followed by Y".


In [12]:
# --- Alternation & Backreferences ---
import re

# Alternation
words = "I have a cat and a dog and a bird"
print(re.findall(r"cat|dog", words))

# Doubled words (backreference)
text = "The the quick brown fox fox jumped"
doubles = re.findall(r"\b(\w+) \1\b", text, re.IGNORECASE)
print(f"Doubled words: {doubles}")

# Date reformatting with backreferences in sub
us_dates = "Meeting on 01-15-2024 and deadline 06-30-2025"
iso = re.sub(r"(\d{2})-(\d{2})-(\d{4})", r"\3-\1-\2", us_dates)
print(f"ISO dates: {iso}")

# Lookahead — get numbers followed by %
text2 = "CPU 85% RAM 60% disk 500GB"
print("Percentages:", re.findall(r"\d+(?=%)", text2))


['cat', 'dog']
Doubled words: ['The', 'fox']
ISO dates: Meeting on 2024-01-15 and deadline 2025-06-30
Percentages: ['85', '60']


---
## 12. `re.sub` — Replace with Power

`re.sub(pattern, replacement, text)` returns a **new string** with replacements.

### Replacement Can Be:

**1. A string** (with backreferences `\1`, `\2`):
```python
re.sub(r"(\w+)@(\w+)", r"\1 [at] \2", email)
```

**2. A function** (receives Match object, returns replacement string):
```python
def mask_email(match):
    name = match.group(1)
    return name[0] + "***@" + match.group(2)

re.sub(r"(\w+)@(\w+\.\w+)", mask_email, text)
```

**3. Count limit:**
```python
re.sub(r"\s+", " ", text, count=1)  # replace only first occurrence
```


In [13]:
# --- Advanced re.sub ---
import re

# Mask emails with a function
text = "Contact alice@email.com or bob@company.co.uk for help"

def mask_email(m):
    user, domain = m.group(1), m.group(2)
    return f"{user[0]}***@{domain}"

masked = re.sub(r"([\w.]+)@([\w.]+)", mask_email, text)
print(masked)

# Normalize whitespace
messy = "Too    many   spaces\t\tand\ttabs"
clean = re.sub(r"\s+", " ", messy).strip()
print(f"Cleaned: '{clean}'")

# Title-case names from "LAST, FIRST" format
names = "SMITH, JOHN\nDOE, JANE"
def flip_name(m):
    last, first = m.group(1), m.group(2)
    return f"{first.title()} {last.title()}"

fixed = re.sub(r"([A-Z]+), ([A-Z]+)", flip_name, names)
print(fixed)


Contact a***@email.com or b***@company.co.uk for help
Cleaned: 'Too many spaces and tabs'
John Smith
Jane Doe


---
## 13. `re.compile` — Compile Once, Use Many Times

If you use the same pattern repeatedly (e.g., in a loop over millions of rows), **compile it first**:

```python
EMAIL_RE = re.compile(r'[\w.+]+@[\w.]+\.[a-z]{2,}', re.IGNORECASE)

for row in millions_of_rows:
    if EMAIL_RE.search(row):
        ...
```

Compiled patterns have the same methods: `.search()`, `.findall()`, `.sub()`, etc.

**When to compile:**
- Pattern used 10+ times → compile
- One-off check → `re.search()` directly is fine


In [14]:
# --- re.compile ---
import re
import time

text = "alice@email.com " * 10000
pattern_str = r"[\w.+]+@[\w.]+\.[a-z]{2,}"

# Compiled
compiled = re.compile(pattern_str, re.IGNORECASE)

start = time.perf_counter()
for _ in range(100):
    compiled.findall(text)
compiled_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(100):
    re.findall(pattern_str, text, re.IGNORECASE)
direct_time = time.perf_counter() - start

print(f"Compiled: {compiled_time:.4f}s")
print(f"Direct:   {direct_time:.4f}s")
print("(Compiled wins more as repetitions grow)")


Compiled: 0.6934s
Direct:   0.5975s
(Compiled wins more as repetitions grow)


---
## 14. Real-World Patterns (Practical Recipes)

> ⚠️ **Honest disclaimer:** "Perfect" email/URL regex is surprisingly hard. For production validation, combine regex with dedicated libraries. These patterns are **practical for data cleaning**, not security gates.

| Task | Pattern | Notes |
|------|---------|-------|
| Email (practical) | `r'[\w.+]+@[\w.-]+\.[a-zA-Z]{2,}'` | Good for extraction |
| Phone (US) | `r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'` | Common formats |
| Date ISO | `r'\d{4}-\d{2}-\d{2}'` | 2024-01-15 |
| Date US | `r'\d{1,2}/\d{1,2}/\d{4}'` | 1/15/2024 |
| IPv4 | `r'\b(?:\d{1,3}\.){3}\d{1,3}\b'` | 192.168.1.1 |
| URL (simple) | `r'https?://[\w./?=&%-]+'` | Basic extraction |
| Currency | `r'\$[\d,]+\.\d{2}'` | $1,234.56 |
| Hashtag | `r'#\w+'` | #Python |
| Mention | `r'@\w+'` | @username |

### Data Cleaning Pipeline Example

```python
# 1. Extract → 2. Validate → 3. Transform → 4. Replace
raw = "  HELLO World  "
step1 = raw.strip()
step2 = re.sub(r"\s+", " ", step1)
step3 = step2.lower()
```


In [15]:
# --- Real-World Pattern Matching ---
import re

messy_data = '''
Contact: alice@email.com, phone 555-123-4567
Backup: bob@company.co.uk | (555) 987-6543
Visit https://example.com/path?q=1 for $1,234.56
Server 192.168.1.100 ERROR at 2024-01-15 14:30:22
Tweet: Learning #Python @instructor today!
'''

print("Emails:  ", re.findall(r'[\w.+]+@[\w.-]+\.[a-zA-Z]{2,}', messy_data))
print("Phones:  ", re.findall(r'\b\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b', messy_data))
print("URLs:    ", re.findall(r'https?://[\w./?=&%-]+', messy_data))
print("IPs:     ", re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', messy_data))
print("Dates:   ", re.findall(r'\d{4}-\d{2}-\d{2}', messy_data))
print("Money:   ", re.findall(r'\$[\d,]+\.\d{2}', messy_data))
print("Hashtags:", re.findall(r'#\w+', messy_data))
print("Mentions:", re.findall(r'@\w+', messy_data))


Emails:   ['alice@email.com', 'bob@company.co.uk']
Phones:   ['555-123-4567', '555) 987-6543']
URLs:     ['https://example.com/path?q=1']
IPs:      ['192.168.1.100']
Dates:    ['2024-01-15']
Money:    ['$1,234.56']
Hashtags: ['#Python']
Mentions: ['@email', '@company', '@instructor']


---
## 15. Case Study — Parsing a Log File

Real data science work often starts with **messy text logs**. Let's parse structured data from unstructured lines.

**Log format:**
```
TIMESTAMP LEVEL message text here
```


In [16]:
# --- Log File Parser (Case Study) ---
import re
from collections import Counter

log_data = """2024-01-15 08:30:15 INFO Server started on 192.168.1.100
2024-01-15 08:31:22 INFO User login from 10.0.0.45
2024-01-15 08:35:01 WARNING High memory usage: 85%
2024-01-15 08:40:55 ERROR Connection failed from 192.168.1.200
2024-01-15 08:42:10 INFO Request processed for 10.0.0.45
2024-01-15 08:45:30 ERROR Database timeout from 192.168.1.100
2024-01-15 08:50:00 WARNING Disk space low: 10%
2024-01-15 08:55:15 INFO Server response time: 250ms"""

LOG_PATTERN = re.compile(
    r'(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})'
    r' (?P<level>\w+)'
    r' (?P<message>.+)'
)

IP_PATTERN = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')

entries = []
for line in log_data.strip().split('\n'):
    m = LOG_PATTERN.match(line)
    if m:
        entry = m.groupdict()
        entry['ips'] = IP_PATTERN.findall(entry['message'])
        entries.append(entry)

# Analysis
print(f"Total log entries: {len(entries)}")
print(f"\nLog level counts:")
for level, count in Counter(e['level'] for e in entries).items():
    print(f"  {level:8s}: {count}")

print(f"\nERROR messages:")
for e in entries:
    if e['level'] == 'ERROR':
        print(f"  [{e['timestamp']}] {e['message']}")

all_ips = [ip for e in entries for ip in e['ips']]
print(f"\nAll IP addresses: {all_ips}")
print(f"Unique IPs: {sorted(set(all_ips))}")


Total log entries: 8

Log level counts:
  INFO    : 4
  ERROR   : 2

ERROR messages:
  [2024-01-15 08:40:55] Connection failed from 192.168.1.200
  [2024-01-15 08:45:30] Database timeout from 192.168.1.100

All IP addresses: ['192.168.1.100', '10.0.0.45', '192.168.1.200', '10.0.0.45', '192.168.1.100']
Unique IPs: ['10.0.0.45', '192.168.1.100', '192.168.1.200']


---
## 16. Regex in Data Science & NLP

| Use Case | Example |
|----------|---------|
| **Clean column names** | `"First Name"` → `"first_name"` |
| **Extract numbers** | `"95kg"` → `95` |
| **Filter rows** | Keep rows matching a pattern |
| **Text features** | Count hashtags, mentions, digits |
| **PII masking** | Redact emails/phones before sharing data |
| **Date parsing** | Pull dates from free-text fields |

### With Pandas (Preview — Sessions 9–10)

```python
import pandas as pd

df['email_domain'] = df['email'].str.extract(r'@([\w.]+)')
df['has_digit'] = df['text'].str.contains(r'\d', regex=True)
df['clean'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)
```

Pandas `.str` methods accept regex patterns when `regex=True` (default).


In [17]:
# --- Data Science Style Cleaning ---
import re

# Clean column names (very common in real datasets!)
raw_columns = [
    'First Name', 'Last  Name', 'Email-Address',
    'Phone #', 'Annual Salary ($)', 'Hire Date'
]

def clean_column_name(col):
    col = col.strip().lower()
    col = re.sub(r'[^\w]', '_', col)      # non-word → underscore
    col = re.sub(r'_+', '_', col)           # collapse multiple _
    return col.strip('_')

clean_cols = [clean_column_name(c) for c in raw_columns]
print("Original:", raw_columns)
print("Cleaned: ", clean_cols)

# Extract numeric value from messy strings
weights = ["95kg", "82.5 kg", "unknown", "70KG", "  100  kg  "]
pattern = re.compile(r'(\d+\.?\d*)\s*kg', re.IGNORECASE)
for w in weights:
    m = pattern.search(w)
    print(f"  {w:15s} → {m.group(1) if m else 'N/A'}")


Original: ['First Name', 'Last  Name', 'Email-Address', 'Phone #', 'Annual Salary ($)', 'Hire Date']
Cleaned:  ['first_name', 'last_name', 'email_address', 'phone', 'annual_salary', 'hire_date']
  95kg            → 95
  82.5 kg         → 82.5
  unknown         → N/A
  70KG            → 70
    100  kg       → 100


---
## 17. Regex vs String Methods — When to Use What

| Task | Use String Methods | Use Regex |
|------|-------------------|-----------|
| Remove whitespace | `s.strip()` | Overkill |
| Split on comma | `s.split(',')` | Overkill |
| Replace exact word | `s.replace('cat', 'dog')` | Overkill |
| Check prefix/suffix | `s.startswith()`, `s.endswith()` | Overkill |
| Find any digit sequence | ❌ awkward loop | `re.findall(r'\d+', s)` |
| Validate email format | ❌ many checks | `re.fullmatch(...)` |
| Multiple separators | ❌ messy | `re.split(r'[,;|]', s)` |
| Mask all emails in text | ❌ very hard | `re.sub(...)` |

**Rule of thumb:** If you can do it cleanly with `.strip()`, `.split()`, `.replace()` — do that. Reach for regex when the **pattern** matters, not an exact string.


In [18]:
# --- Regex vs String Methods ---
text = "  hello,world;python|data  "

# Simple — use string methods
print(text.strip())
print("hello world".replace(",", " "))

# Complex — regex wins
print(re.split(r'[,;|\s]+', text.strip()))

# Validation — regex wins
emails = ["alice@test.com", "bad@", "@bad.com", "bob@co.uk"]
pat = re.compile(r'^[\w.+]+@[\w.-]+\.[a-zA-Z]{2,}$')
for e in emails:
    print(f"  {e:20s} {'✅' if pat.fullmatch(e) else '❌'}")


hello,world;python|data
hello world
['hello', 'world', 'python', 'data']
  alice@test.com       ✅
  bad@                 ❌
  @bad.com             ❌
  bob@co.uk            ✅


---
## 18. Debugging Regex & Common Mistakes

### Top 10 Beginner Mistakes

| # | Mistake | Fix |
|---|---------|-----|
| 1 | Forgetting `r"..."` | Always use raw strings |
| 2 | `.` instead of `\.` for literal dot | Escape metacharacters |
| 3 | Greedy `.*` swallowing too much | Use `.*?` or be specific |
| 4 | `match` vs `search` confusion | `search` finds anywhere |
| 5 | Not checking for `None` | `if m: m.group()` |
| 6 | `findall` with groups returns groups | Use `finditer` or `(?:...)` |
| 7 | Case sensitivity | Add `re.IGNORECASE` |
| 8 | `^`/`$` without `re.MULTILINE` | Add flag for line-by-line |
| 9 | Over-complicated patterns | Build incrementally, test each part |
| 10 | Using regex for HTML/JSON | Use `BeautifulSoup`, `json` module |

### Debugging Workflow

1. **Start simple** — test `r"\d+"` before building `r"^(\d{3})-..."`
2. **Test interactively** — run cells, print `match.group()` and `match.span()`
3. **Use online tools** — [regex101.com](https://regex101.com) (select Python flavor)
4. **Use `re.VERBOSE`** — add comments to complex patterns
5. **Print the pattern** — `print(repr(pattern))` to spot escaping issues

### Reading Error Messages

```python
re.search(r'[', "test")   # re.error: unterminated character set
re.search(r'(abc', "test") # re.error: missing ), unterminated subpattern
```


In [19]:
# --- Debugging Demo ---
import re

# Build a pattern step by step
text = "Order #1234 on 2024-01-15 for $99.99"

# Step 1: find numbers
print("Step 1:", re.findall(r"\d+", text))

# Step 2: find date specifically
print("Step 2:", re.findall(r"\d{4}-\d{2}-\d{2}", text))

# Step 3: find price with $
print("Step 3:", re.findall(r"\$\d+\.\d{2}", text))

# repr() shows exactly what Python sees
pat = r"\d{4}-\d{2}-\d{2}"
print(f"\nPattern repr: {repr(pat)}")


Step 1: ['1234', '2024', '01', '15', '99', '99']
Step 2: ['2024-01-15']
Step 3: ['$99.99']

Pattern repr: '\\d{4}-\\d{2}-\\d{2}'


---
## 19. Quick Reference Cheat Sheet

### Metacharacters
```
.   any char     ^   start       $   end
*   0+ times     +   1+ times    ?   0 or 1
{n} exactly n   {n,m} n to m    \   escape
|   or           ()  group       []  char class
```

### Shorthands
```
\d digit   \D non-digit   \w word char   \W non-word
\s space   \S non-space   \b word boundary
```

### Functions
```
re.search()    first match anywhere
re.match()     match at start only
re.fullmatch() entire string must match
re.findall()   all matches as list
re.finditer()  all matches as objects
re.sub()       replace
re.split()     split
re.compile()   pre-compile
```

### Flags
```
re.I  ignore case    re.M  multiline    re.S  dot matches newline
re.X  verbose        re.A  ASCII only
```


---
## 20. Practice Exercises

Work through these in order. Each builds on the previous.

### Exercise 1: Extract All Numbers
From the text below, extract **all integers** (positive and negative) and print them as a list.


In [20]:
# Exercise 1
text = "Temperatures: -5, 0, 12, and -3 degrees. Elevation: 1500m, depth: -200m"
# Your code here:


### Exercise 2: Validate Indian Mobile Numbers
Write a function `is_valid_indian_mobile(s)` that returns `True` if `s` is exactly a 10-digit number starting with 6, 7, 8, or 9.
Test with: `"9876543210"`, `"5876543210"`, `"98765"`, `"98765432101"`


In [21]:
# Exercise 2
def is_valid_indian_mobile(s):
    # Your code here
    pass

tests = ["9876543210", "5876543210", "98765", "98765432101"]
for t in tests:
    print(f"  {t} → {is_valid_indian_mobile(t)}")


  9876543210 → None
  5876543210 → None
  98765 → None
  98765432101 → None


### Exercise 3: Password Strength Checker
A valid password must:
- Be 8–20 characters long
- Contain at least one uppercase letter, one lowercase, one digit, and one special char from `@#$%!`

Write `check_password(pw)` returning `(is_valid, list_of_missing_rules)`.


In [22]:
# Exercise 3
def check_password(pw):
    # Your code here
    pass

for pw in ["Abc@1234", "weak", "NoDigits!", "ALLUPPER1@"]:
    print(f"  {pw:15s} → {check_password(pw)}")


  Abc@1234        → None
  weak            → None
  NoDigits!       → None
  ALLUPPER1@      → None


### Exercise 4: Extract and Reformat Dates
Convert all `MM/DD/YYYY` dates in the text to `YYYY-MM-DD` format.


In [23]:
# Exercise 4
text = "Meeting 03/15/2024, deadline 12/01/2024, holiday 01/01/2025"
# Your code here:


### Exercise 5: Log Analyzer
From the log data below:
1. Count each log level (INFO, WARNING, ERROR)
2. Extract all IP addresses
3. List all ERROR messages
4. Find the average response time (extract number before `ms` from the last line)


In [24]:
# Exercise 5
log = """2024-01-15 08:30:15 INFO Server started on 192.168.1.100
2024-01-15 08:35:01 WARNING High memory usage: 85%
2024-01-15 08:40:55 ERROR Connection failed from 192.168.1.200
2024-01-15 08:45:30 ERROR Database timeout from 192.168.1.100
2024-01-15 08:55:15 INFO Server response time: 250ms"""

# Your code here:


### Exercise 6: Data Cleaner (Mini Project)
Given the messy customer records below, write a pipeline that:
1. Extracts name, email, and phone from each line (formats vary!)
2. Normalizes phone to `XXX-XXX-XXXX`
3. Masks email as `a***@domain.com`
4. Returns a list of dicts


In [25]:
# Exercise 6
records = [
    "John Smith | john.smith@email.com | (555) 123-4567",
    "Jane Doe, jane@company.co.uk, 555-987-6543",
    "Bob Lee  bob@mail.org  555.111.2222",
]
# Your code here:


---
## 21. Exercise Solutions

<details>
<summary>Click to expand solutions (try exercises first!)</summary>

Run the cell below after attempting the exercises yourself.
</details>


In [26]:
# --- SOLUTIONS (try exercises first!) ---
import re
from collections import Counter

# Exercise 1
text = "Temperatures: -5, 0, 12, and -3 degrees. Elevation: 1500m, depth: -200m"
print("Ex1:", re.findall(r'-?\d+', text))

# Exercise 2
def is_valid_indian_mobile(s):
    return bool(re.fullmatch(r'[6-9]\d{9}', s))

for t in ["9876543210", "5876543210", "98765", "98765432101"]:
    print(f"  Ex2 {t} → {is_valid_indian_mobile(t)}")

# Exercise 3
def check_password(pw):
    rules = []
    if not re.search(r'.{8,20}', pw):
        rules.append("length 8-20")
    if not re.search(r'[A-Z]', pw):
        rules.append("uppercase")
    if not re.search(r'[a-z]', pw):
        rules.append("lowercase")
    if not re.search(r'\d', pw):
        rules.append("digit")
    if not re.search(r'[@#$%!]', pw):
        rules.append("special char")
    return (len(rules) == 0, rules)

for pw in ["Abc@1234", "weak", "NoDigits!", "ALLUPPER1@"]:
    print(f"  Ex3 {pw:15s} → {check_password(pw)}")

# Exercise 4
text = "Meeting 03/15/2024, deadline 12/01/2024, holiday 01/01/2025"
ex4 = re.sub(r'(\d{2})/(\d{2})/(\d{4})', r'\3-\1-\2', text)
print(f"Ex4: {ex4}")

# Exercise 5
log = """2024-01-15 08:30:15 INFO Server started on 192.168.1.100
2024-01-15 08:35:01 WARNING High memory usage: 85%
2024-01-15 08:40:55 ERROR Connection failed from 192.168.1.200
2024-01-15 08:45:30 ERROR Database timeout from 192.168.1.100
2024-01-15 08:55:15 INFO Server response time: 250ms"""

levels = re.findall(r'\b(INFO|WARNING|ERROR)\b', log)
print(f"Ex5 levels: {Counter(levels)}")
print(f"Ex5 IPs: {re.findall(r'(?:\d{1,3}\.){3}\d{1,3}', log)}")
errors = re.findall(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} ERROR (.+)', log)
print(f"Ex5 errors: {errors}")
times = [int(x) for x in re.findall(r'(\d+)ms', log)]
print(f"Ex5 avg ms: {sum(times)/len(times)}")

# Exercise 6
records = [
    "John Smith | john.smith@email.com | (555) 123-4567",
    "Jane Doe, jane@company.co.uk, 555-987-6543",
    "Bob Lee  bob@mail.org  555.111.2222",
]

def clean_records(records):
    results = []
    email_pat = re.compile(r'([\w.]+)@([\w.]+)')
    phone_pat = re.compile(r'\(?(\d{3})\)?[-.\s]?(\d{3})[-.\s]?(\d{4})')
    for line in records:
        email_m = email_pat.search(line)
        phone_m = phone_pat.search(line)
        if email_m:
            name_part = line[:email_m.start()]
            name = re.sub(r'[,|]', '', name_part).strip()
            name = re.sub(r'\s+', ' ', name)
            masked = f"{email_m.group(1)[0]}***@{email_m.group(2)}"
        else:
            name, masked = None, None
        if phone_m:
            phone = f"{phone_m.group(1)}-{phone_m.group(2)}-{phone_m.group(3)}"
        else:
            phone = None
        results.append({"name": name, "email_masked": masked, "phone": phone})
    return results

print(f"Ex6: {clean_records(records)}")


Ex1: ['-5', '0', '12', '-3', '1500', '-200']
  Ex2 9876543210 → True
  Ex2 5876543210 → False
  Ex2 98765 → False
  Ex2 98765432101 → False
  Ex3 Abc@1234        → (True, [])
  Ex3 weak            → (False, ['length 8-20', 'uppercase', 'digit', 'special char'])
  Ex3 NoDigits!       → (False, ['digit'])
  Ex3 ALLUPPER1@      → (False, ['lowercase'])
Ex4: Meeting 2024-03-15, deadline 2024-12-01, holiday 2025-01-01
Ex5 levels: Counter({'INFO': 2, 'ERROR': 2, 'WARNING': 1})
Ex5 IPs: ['192.168.1.100', '192.168.1.200', '192.168.1.100']
Ex5 errors: ['Connection failed from 192.168.1.200', 'Database timeout from 192.168.1.100']
Ex5 avg ms: 250.0
Ex6: [{'name': 'John Smith', 'email_masked': 'j***@email.com', 'phone': '555-123-4567'}, {'name': 'Jane Doe', 'email_masked': 'j***@company.co.uk', 'phone': '555-987-6543'}, {'name': 'Bob Lee', 'email_masked': 'b***@mail.org', 'phone': '555-111-2222'}]


---
## 📝 Session 4B Summary

### What You Learned

| Topic | Key Takeaway |
|-------|--------------|
| **Mental model** | Regex describes text *patterns*, not exact strings |
| **Raw strings** | Always `r"..."` — avoids double-backslash confusion |
| **Metacharacters** | `. ^ $ * + ? { } [ ] ( ) \|` are the building blocks |
| **Character classes** | `[abc]`, `[0-9]`, `[^abc]` for custom character sets |
| **Quantifiers** | `* + ? {n,m}` — watch greedy vs lazy `*?` |
| **Anchors** | `^ $ \b` for position; use `fullmatch` for validation |
| **Groups** | `()` to capture; `(?P<name>)` for named groups |
| **re toolkit** | `search`, `findall`, `sub`, `split`, `compile` |
| **Flags** | `IGNORECASE`, `MULTILINE`, `DOTALL`, `VERBOSE` |
| **Real world** | Logs, emails, phones, column cleaning, PII masking |

### Key Gotchas to Remember
- Always use **raw strings** `r'...'` for patterns
- **`search`** finds anywhere; **`match`** only at start; **`fullmatch`** validates entire string
- **Greedy** `.*` eats too much — use `.*?` when needed
- **`findall` with groups** returns groups, not full matches
- Check for **`None`** before calling `.group()`
- Use **string methods** for simple tasks; regex for patterns
- Don't parse HTML/JSON with regex — use proper libraries

### What's Next
- **Session 5**: Modules, Packages, `pip`, Virtual Environments
- **Sessions 9–10**: Pandas `.str` accessor uses regex heavily — you'll reuse everything from this session!

> 🎯 **Practice tip:** Spend 15 minutes on [regex101.com](https://regex101.com) with Python flavor selected. Type a pattern, paste sample text, and watch matches highlight in real time. This is the fastest way to build regex intuition.
